# HFB, Gaussian fidelity, and angular-momentum projection

This notebook is a small, runnable guide to the main data classes in `NSMFermions`. It uses a four-mode proton-neutron model, so every calculation finishes quickly while following the same path as the CKI $^{8}$Be benchmark.

The central flow is

$$
|\Phi\rangle\; (\texttt{HFBState})
\longrightarrow
\sum_q w_q T_q|\Phi\rangle\; (\texttt{BogoliubovVacuumSeries})
\longrightarrow
E,F,|\Psi_{NZJ=0}\rangle\; (\texttt{NumberProjectionResult}).
$$

A dataclass is only a named container. The physics is implemented by the constructors and methods that create and consume these objects.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

# Work both from the repository root and from a notebook subdirectory.
ROOT = Path.cwd()
if not (ROOT / 'src' / 'NSMFermions').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src' / 'NSMFermions'))
sys.path.insert(0, str(ROOT / 'tests'))

from hfb import HFBHamiltonian, HFBState, BogoliubovVacuumSeries, solve_hfb
from angular_momentum import (
    J0Grid, single_particle_angular_momentum, euler_rotation,
    polynomial_j0_grid, ParticleNumberJ0ProjectedEnergy,
    project_state_observables,
)
from number_projection import NumberProjectionResult, exact_ground_state
from gaussian_fidelity import (
    GaussianFidelityResult, SlaterFidelityResult,
    maximize_gaussian_fidelity, maximize_slater_fidelity,
)
from test_number_projection import FermiHubbardHamiltonian


## 1. A solvable four-mode model

Each one-body mode is encoded as $(n,l,j,m,t,t_z)$. The first two modes below are proton modes and the last two are neutron modes. The interaction is an attractive projector onto the proton-neutron $J=0$ pair

$$
|P_0\rangle=\frac{|p_+n_-\rangle-|p_-n_+\rangle}{\sqrt2}.
$$

In [ ]:
states = [
    (0, 0, 0.5, -0.5, 0.5,  0.5),  # proton m=-1/2
    (0, 0, 0.5,  0.5, 0.5,  0.5),  # proton m=+1/2
    (0, 0, 0.5, -0.5, 0.5, -0.5),  # neutron m=-1/2
    (0, 0, 0.5,  0.5, 0.5, -0.5),  # neutron m=+1/2
]
neutron_modes = [2, 3]
targets = [1, 1]

pair = np.zeros((4, 4), complex)
pair[1, 2] = 1 / np.sqrt(2)
pair[0, 3] = -1 / np.sqrt(2)
pair -= pair.T
interaction = -np.einsum('ij,kl->ijkl', pair, pair.conj())
ham = HFBHamiltonian(np.zeros((4, 4)), interaction)
fixed_sector_ham = FermiHubbardHamiltonian(ham, neutron_modes, targets)
exact_energy, exact_target = exact_ground_state(fixed_sector_ham)
print('fixed-(N,Z) basis:', fixed_sector_ham.occupations)
print('exact energy:', exact_energy)
print('exact coefficients:', exact_target)


## 2. `HFBState`: one intrinsic Gaussian vacuum

For a finite antisymmetric Thouless matrix $Z$,

$$
|\Phi(Z)\rangle=\mathcal N\exp\left(\frac12\sum_{ij}Z_{ij}c_i^\dagger c_j^\dagger\right)|0\rangle.
$$

`HFBState.from_thouless` constructs canonical Bogoliubov matrices $U,V$. Its main observables are

$$\rho=V^*V^T,\qquad \kappa=V^*U^T. $$

In [ ]:
Z = np.zeros((4, 4), complex)
Z[1, 2] = 0.7
Z[0, 3] = -0.3
Z -= Z.T
state = HFBState.from_thouless(Z)

print('U shape:', state.U.shape, 'V shape:', state.V.shape)
print('canonical error:', state.canonical_error())
occupations_1b = np.diag(state.rho).real
proton_modes = [i for i in range(len(states)) if i not in neutron_modes]
print('<N>, <Z>:',
      [occupations_1b[neutron_modes].sum(), occupations_1b[proton_modes].sum()])
print('||kappa||:', np.linalg.norm(state.kappa))
print('probability of the N=1,Z=1 sector:',
      state.fixed_sector_weight(fixed_sector_ham.occupations))
print('raw fidelity with the exact target:',
      state.fixed_sector_fidelity(exact_target, fixed_sector_ham.occupations))
print('fidelity after normalizing the selected sector:',
      state.fixed_sector_fidelity(
          exact_target, fixed_sector_ham.occupations, projected=True))


The distinction between the last two numbers is important. The raw fidelity includes the probability that the intrinsic vacuum actually occupies the requested sector. The projected fidelity first constructs and normalizes $P_NP_Z|\Phi\rangle$.

## 3. `J0Grid`: an exact finite-space Euler quadrature

The spatial projector is

$$P_0=\frac1{8\pi^2}\int d\alpha\,d\gamma\,d(\cos\beta)\;R(\alpha,\beta,\gamma).$$

The code first bounds every possible many-body magnetic projection $M=\sum_i m_i$. Periodic $\alpha$ and $\gamma$ grids resolve the Fourier factors $e^{-iM\alpha}$ and $e^{-iK\gamma}$. After they select $M=K=0$, $d^J_{00}(\beta)=P_J(\cos\beta)$, so Gauss-Legendre quadrature integrates the remaining Legendre polynomials exactly.

In [ ]:
jx, jy, jz = single_particle_angular_momentum(states)
print('||[Jx,Jy]-iJz||:', np.linalg.norm(jx @ jy - jy @ jx - 1j*jz))

grid = polynomial_j0_grid(states, neutron_modes, targets)
print('M bound:', grid.m_bound, 'J bound:', grid.j_bound)
print('(L_alpha, L_beta, L_gamma):',
      (len(grid.alpha), len(grid.cos_beta), len(grid.gamma)))
print('Euler rotations:', grid.size)

R = euler_rotation(0.3, 0.7, -0.2, (jx, jy, jz))
print('rotation unitarity error:', np.linalg.norm(R.conj().T @ R - np.eye(4)))


For this model, one neutron plus one proton gives $M_{\rm bound}=1$ and $J_{\rm bound}=1$. Therefore the exact default grid is $3\times1\times3=9$ Euler rotations. `frozen=True` on `J0Grid` prevents accidentally replacing fields after the nodes have been constructed.

## 4. `BogoliubovVacuumSeries`: keep the projector as rotated vacua

The combined projector is represented as

$$
P_NP_ZP_0|\Phi\rangle\approx\sum_q w_qT_q|\Phi\rangle.
$$

A `BogoliubovVacuumSeries` stores the common intrinsic state, every one-body transformation $T_q$, every complex weight $w_q$, and the grid metadata. It deliberately does not build a determinant-basis vector yet.

In [ ]:
projector = ParticleNumberJ0ProjectedEnergy(
    ham, states, neutron_modes, targets
)
series = projector.projected_series(state)

assert isinstance(series, BogoliubovVacuumSeries)
print('projection:', series.projection)
print('number grid:', series.number_grid)
print('Euler grid:', series.euler_grid)
print('number of transformed vacua:', series.number_of_vacua)
print('transform array shape:', series.transformations.shape)
print('weight array shape:', series.weights.shape)


The default number grids contain three neutron and three proton gauge angles. Together with nine Euler rotations this gives $3\times3\times9=81$ vacuum terms. These terms must be added coherently: sum complex amplitudes first, and normalize only afterward.

## 5. `NumberProjectionResult`: observables at the basis boundary

`project_state_observables` finally expands the coherent series in the fixed-$(N,Z)$ determinant ordering. It returns the normalized projected vector, its energy, its norm before normalization, and an optional target fidelity.

In [ ]:
result = project_state_observables(series, fixed_sector_ham, exact_target)
assert isinstance(result, NumberProjectionResult)
kernel_energy = projector.series_energy(series)

print('projected norm:', result.sector_weight)
print('projected vector:', result.projected_vector)
print('basis energy:', result.energy)
print('transition-kernel energy:', kernel_energy)
print('exact energy:', exact_energy)
print('fidelity with exact J=0 target:', result.fidelity)


In this solvable model the projected state has fidelity one with the exact ground state. The transition-kernel energy and determinant-basis energy are two evaluations of the same stored vacuum series and should agree to numerical precision.

## 6. Gaussian-fidelity result classes

`maximize_gaussian_fidelity` searches finite Thouless matrices and returns `GaussianFidelityResult`. `maximize_slater_fidelity` searches the number-conserving Slater boundary and returns `SlaterFidelityResult`. The two searches are separate because a non-vacuum Slater determinant has singular $U$ and lies at infinite Thouless norm.

In [ ]:
interior = maximize_gaussian_fidelity(
    fixed_sector_ham, exact_target, starts=2, seed=9, maxiter=300
)
boundary = maximize_slater_fidelity(
    fixed_sector_ham, exact_target, starts=3, seed=2, maxiter=500
)
assert isinstance(interior, GaussianFidelityResult)
assert isinstance(boundary, SlaterFidelityResult)

print('finite-Z Gaussian fidelity:', interior.fidelity)
print('finite-Z converged:', interior.converged,
      'gradient norm:', interior.gradient_norm)
print('Slater-boundary fidelity:', boundary.fidelity)
print('Slater converged:', boundary.converged,
      'gradient norm:', boundary.gradient_norm)
print('best-found Gaussian fidelity:',
      max(interior.fidelity, boundary.fidelity))


These are raw intrinsic fidelities, not projected fidelities. For a finite-$Z$ result you can inspect both with

```python
interior.state.fixed_sector_fidelity(target, occupations)
interior.state.fixed_sector_fidelity(target, occupations, projected=True)
```

## 7. Why two nearly orthogonal intrinsic states can have the same ground-state fidelity

If the exact ground state has $J=0$, then $\langle\Psi_0|R(\Omega)=\langle\Psi_0|$. Consequently

$$|\langle\Psi_0|R(\Omega)|\Phi\rangle|^2=|\langle\Psi_0|\Phi\rangle|^2.$$

Every orientation of an intrinsic Gaussian is therefore equally good according to the ground-state-fidelity objective. Two independently optimized states may choose almost orthogonal orientations. The diagnostic quantity is

$$F_{\rm aligned}=\max_{\alpha,\beta,\gamma}|\langle\Phi_A|R(\alpha,\beta,\gamma)|\Phi_B\rangle|^2.$$

In [ ]:
diagnostic_path = ROOT / 'benchmarks' / 'results' / 'cki_be8_rotation_equivalence.json'
if diagnostic_path.exists():
    diagnostic = json.loads(diagnostic_path.read_text(encoding='utf-8'))
    for key in (
        'hfb_ground_fidelity',
        'closest_gaussian_ground_fidelity',
        'raw_mutual_fidelity',
        'best_aligned_fidelity',
    ):
        print(f'{key}: {diagnostic[key]}')
else:
    print('Run: python benchmarks/test_rotation_equivalence.py')


For the stored CKI $^{8}$Be calculation, the two ground-state fidelities are approximately $0.20181$ and $0.20215$. Their raw mutual fidelity is $7.60\times10^{-4}$, while spatial alignment raises it to $0.99743$. Thus they are almost the same intrinsic Slater shape in different spatial orientations.

## 8. Running the full CKI workflow

From the repository root:

```powershell
python benchmarks/cki_be8_pav.py
python benchmarks/cki_be8_j0_projection.py
python benchmarks/cki_be8_best_gaussian.py
python benchmarks/test_rotation_equivalence.py
```

For CKI $^{8}$Be the exact default Euler grid is $9\times4\times9=324$ rotations. Combined with the $7\times7$ particle-number grid, the projected series has $15{,}876$ vacua. The implementation currently supports scalar $J=0$ projection. General $J>0$ requires the full $P^J_{MK}$ matrices and $K$ mixing.

## 9. Why the Pfaffian overlap has a sign prefactor

For the block ordering used by the code, the normalized overlap is

$$
\langle\Phi|T|\Phi\rangle=\frac{(-1)^{m(m+1)/2}}{\sqrt{\det(I+Z^\dagger Z)}}\operatorname{pf}\begin{pmatrix}TZT^T&-I\\I&-Z^*\end{pmatrix}.
$$

The factor $(-1)^{m(m+1)/2}$ is a convention correction caused by reordering fermionic operators into the displayed block order; it is not an additional physical phase. At $Z=0$ and $T=I$, the raw block-matrix Pfaffian is $(-1)^{m(m+1)/2}$, while the vacuum must overlap itself by $+1$, so the prefactor supplies the required normalization check.

The older determinant expression determines only $\langle\Phi|T|\Phi\rangle^2$ and therefore leaves a square-root sign/phase ambiguity. Once the single-particle ordering is fixed, the Pfaffian is a definite polynomial and preserves the phase continuously.

In [ ]:
from gauge_projection import pfaffian, transformed_vacuum_overlap

m = 2
vacuum_block = np.block([
    [np.zeros((m, m)), -np.eye(m)],
    [np.eye(m), np.zeros((m, m))],
])
convention_sign = (-1)**(m*(m+1)//2)
print('raw Pfaffian:', pfaffian(vacuum_block))
print('convention sign:', convention_sign)
print('corrected vacuum overlap:', convention_sign * pfaffian(vacuum_block))


## Class summary

| Class | Mathematical meaning | Created by |
|---|---|---|
| `HFBState` | Intrinsic Gaussian vacuum $|\Phi\rangle$ | `from_thouless`, `from_slater`, or `solve_hfb` |
| `HFBResult` | Optimized state plus convergence diagnostics | `solve_hfb` |
| `J0Grid` | Euler nodes, weights, and finite-space bounds | `polynomial_j0_grid` |
| `BogoliubovVacuumSeries` | $\sum_qw_qT_q|\Phi\rangle$ | `projected_series` |
| `NumberProjectionResult` | Normalized projected vector and observables | `project_state_observables` |
| `GaussianFidelityResult` | Best finite-$Z$ Gaussian found | `maximize_gaussian_fidelity` |
| `SlaterFidelityResult` | Best Slater-boundary Gaussian found | `maximize_slater_fidelity` |